# Single Subtitle — Burn

Third step of the pipeline (after `video-base-*.ipynb` and `caption-single-generate.ipynb`): burns the
chosen subtitle file onto the video base as a single, plain caption track.

**Which subtitle file gets used?** `config.nome_legenda_unica` — by default, the Whisper
transcription from `caption-single-generate.ipynb` (`{NOME}_whisper_{IDIOMA}.srt`). If you downloaded it,
corrected it, and re-uploaded it to Drive (same filename), this notebook picks up your
correction automatically — it always reads the current file from Drive, never a stale local
copy from an earlier session.

To use a **different** file instead (e.g. you saved your correction under a new name), set
`NOME_LEGENDA_UNICA` in the Configuration cell below.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg fonts-noto-cjk > /dev/null 2>&1
print('✅ ffmpeg')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")
    print("   Make sure the .py files are in pipeline/modulos/ on Drive.")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as the earlier steps ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY (must match video-base-*.ipynb / caption-single-generate.ipynb) ──
NOME_ORACAO = "40_Matt_02"

# ── 2. NARRATION LANGUAGE (must match caption-single-generate.ipynb) ────────────────
IDIOMA_MESTRE = "en"

# ── 3. WHICH SUBTITLE FILE TO BURN ──────────────────────────────────────────
# Leave blank to use the default (the Whisper transcription from
# caption-single-generate.ipynb: {NOME}_whisper_{IDIOMA}.srt). Fill in only if you want
# to point at a different SRT already saved in the video's folder on Drive
# (e.g. a manual correction saved under a new name).
NOME_LEGENDA_UNICA = ""

# ── 4. VERSE REFERENCE OVERLAY (optional) ───────────────────────────────────
# A small fixed indicator in the top-left corner (e.g. "Matt/Mt/마 2:4") that
# updates only the verse number as the narration advances — separate from
# the subtitle itself. OPTIONAL: only makes sense for verse-by-verse Bible
# study videos — leave INCLUIR_VERSICULO = False for prayer/free-content
# videos that aren't tied to Bible verses.
INCLUIR_VERSICULO = True

# Only used if INCLUIR_VERSICULO = True:
CAPITULO = 1
ABREVIACOES_LIVRO = {"en": "Matt", "pt": "Mt", "es": "Mt", "fr": "Mt", "ko": "마"}
# Paste the chapter text with verse numbers as standalone tokens in the flow
# (e.g. "1 Now when Jesus was born ... 2 Where is he who is born ..."):
TEXTO_VERSICULOS = ""

# ── 5. DRIVE ROOT FOLDER ──────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:           {NOME_ORACAO}")
print(f"   Narration lang:  {IDIOMA_MESTRE}")
print(f"   Subtitle file:   {NOME_LEGENDA_UNICA or '(default — Whisper transcription)'}")
print(f"   Verse overlay:   {'ON — ' + '/'.join(dict.fromkeys(ABREVIACOES_LIVRO.values())) + f' {CAPITULO}:N' if INCLUIR_VERSICULO else 'off'}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from caption_pipeline import CaptionPipeline

config = PipelineConfig(
    NOME_ORACAO         = NOME_ORACAO,
    PASTA_DRIVE_RAIZ     = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE         = IDIOMA_MESTRE,
    NOME_LEGENDA_UNICA    = NOME_LEGENDA_UNICA,
    CAPITULO              = CAPITULO,
    ABREVIACOES_LIVRO     = ABREVIACOES_LIVRO,
)

pipeline = CaptionPipeline(config)

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:           {config.NOME_ORACAO}")
print(f"   Folder:          {config.pasta_oracao}")
print(f"   Subtitle file:   {config.nome_legenda_unica}")
print(f"   Output video:    {config.NOME_VIDEO_FINAL}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📖 GENERATE VERSE REFERENCE (optional — only if INCLUIR_VERSICULO) ║
# ║  Skipped automatically if INCLUIR_VERSICULO = False.              ║
# ╚══════════════════════════════════════════════════════════════════╝

if INCLUIR_VERSICULO:
    if not TEXTO_VERSICULOS.strip():
        raise ValueError("INCLUIR_VERSICULO is True but TEXTO_VERSICULOS is empty — paste the verse-numbered text in the Configuration cell.")
    srt_versiculo = pipeline.gerar_legenda_versiculo(TEXTO_VERSICULOS)
    print(f"✅ Verse reference generated: {srt_versiculo.name}")
else:
    print("Verse reference overlay is off (INCLUIR_VERSICULO = False) — skipping.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔥 LOAD + BURN — single subtitle track onto the video base      ║
# ║  Always downloads the current file from Drive — picks up any     ║
# ║  manual correction automatically.                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

legendas = pipeline.carregar_legenda_unica()
print(f"{len(legendas)} caption blocks loaded from {config.nome_legenda_unica}\n")

video_final = pipeline.queimar_legenda_unica(legendas, incluir_versiculo=INCLUIR_VERSICULO)
print(f"\n✅ Final video: {video_final.name} ({video_final.stat().st_size/1_048_576:.1f} MB)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW FINAL VIDEO                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

from IPython.display import Video, display

display(Video(str(video_final), embed=True, width=800))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — FINAL VIDEO                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

print(f"📥 Downloading {video_final.name} ({video_final.stat().st_size/1_048_576:.1f} MB)...")
files.download(str(video_final))
